In [ ]:
# CELL 1 — Connection and saved-input preparation scope
# Paste private sf_options = {...} here. No credentials are supplied.
# Runtime: Spark Snowflake connector, numpy, pandas, sklearn, torch, matplotlib.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if "sf_options" not in globals() or not isinstance(sf_options, dict) or "spark" not in globals():
    raise RuntimeError("Supply sf_options on the approved Spark runtime.")
sf_options_dl_poc = dict(sf_options)
sf_options_dl_poc.update(sfDatabase="DSVC_TAKEDA_TA_PRIVATE", sfSchema="DS_ML")
PREFIX = "TAK861_TX_READY_V63_DL_POC"
EXPERIMENT_PREFIX = PREFIX + "_TEMPORAL_SELECTION_V1"
REFERENCE_RUN_ID = "RUN_001"
RUN_ID = "P001"  # New ID for changed ranking/training settings or code; same ID in all four notebooks.
PLAN_SEED = 42
import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,15}", RUN_ID):
    raise ValueError("RUN_ID must be a short uppercase identifier.")
PREPARATION_TABLE = EXPERIMENT_PREFIX + "_PREPARATION"
SPLIT_AUDIT_TABLE = EXPERIMENT_PREFIX + "_SPLIT_AUDIT"
REFERENCE_MODEL_TABLE = PREFIX + "_MODEL_" + REFERENCE_RUN_ID
RUN_PREFIX = EXPERIMENT_PREFIX + "_" + RUN_ID
INTERNAL_TABLE = RUN_PREFIX + "_INTERNAL"
RANK_MODEL_TABLE = RUN_PREFIX + "_RANK_MODEL"
SELECTION_TABLE = RUN_PREFIX + "_FEATURE_SELECTION"
MODEL_TABLE = RUN_PREFIX + "_MODEL"
EVALUATION_TABLE = RUN_PREFIX + "_EVALUATION"


PREPARATION_TABLE = EXPERIMENT_PREFIX + "_PREPARATION"
REFERENCE_MODEL_TABLE = PREFIX + "_MODEL_" + REFERENCE_RUN_ID
print("Reuse and validate the completed V63 tensor; no raw claims rebuild.")
print("Original FEATURE_MAP, SNAPSHOTS and TENSOR_MONTHLY tables stay unchanged.")
print("Feature reduction is fitted on TRAIN only in notebook 03.")


In [ ]:
# CELL 2 — Input contracts and aggregate-report helpers
"""Embedded by the delivery notebooks; no repository dependency at runtime."""
import base64
import hashlib
import io
import json
import math
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_sf(suffix):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", f"{PREFIX}_{suffix}").load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def checked_metadata(frame, include_split=False):
    columns = ["PATIENT_ID", "END_DT", "RESP"]
    if include_split:
        columns += ["SPLIT", "SPLIT_CONFIG"]
    out = frame.loc[:, columns].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("PATIENT_ID must retain its original nonempty string value.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("END_DT must be a date without an intraday time/timezone.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("RESP must be exactly 0 or 1 before conversion.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys in snapshot metadata.")
    if include_split:
        if set(out.SPLIT) != {"train", "validation", "test"}:
            raise ValueError("Expected the saved train, validation and test assignments.")
        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():
            raise ValueError("Patient overlap between splits.")
        if len(out.SPLIT_CONFIG.unique()) != 1:
            raise ValueError("The split contains inconsistent creation settings.")
        for _, part in out.groupby("SPLIT", observed=True):
            if set(part.RESP) != {0, 1}:
                raise ValueError("Each split must contain both response classes.")
    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)


def validate_metadata_pair(snapshot_frame, manifest_frame, features):
    source = checked_metadata(snapshot_frame)
    manifest = checked_metadata(manifest_frame, include_split=True)
    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Frozen split and source snapshots differ in keys or labels.")
    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])
    # Match the feature-order hash created in the completed split notebook.
    feature_order_hash = hashlib.sha256(
        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()
    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:
        raise ValueError("Feature order or timesteps differ from the frozen split.")
    return manifest, creation


def new_tensor_buffer(metadata, feature_count, seq_len=12):
    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")
    path = Path(temporary.name) / "counts.float32"
    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))
    return temporary, X


def fill_tensor(X, metadata, sequence_rows):
    """Place rows by canonical keys, independent of Spark partition order."""
    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}
    seen = np.zeros(len(metadata), dtype=bool)
    seq_len, feature_count = X.shape[1:]
    for row in sequence_rows:
        raw_date = row["END_DT"]
        if raw_date is None or row["PATIENT_ID"] is None:
            raise ValueError("Null monthly snapshot key.")
        date = pd.Timestamp(raw_date)
        if date.tzinfo is not None or date != date.normalize():
            raise ValueError("Monthly END_DT is not an exact date.")
        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))
        if key not in positions:
            raise ValueError("Unexpected monthly snapshot key.")
        i = positions[key]
        if seen[i] or row["RESP"] != int(metadata.RESP.iloc[i]):
            raise ValueError("Duplicate monthly snapshot or changed label.")
        sequence = row["SEQUENCE"]
        steps = [month["T"] for month in sequence]
        if len(sequence) != seq_len or any(t is None for t in steps):
            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")
        # Check original values before any integer conversion.
        if sorted(steps) != list(range(seq_len)):
            raise ValueError("Timesteps must be unique integers 0 through 11.")
        sequence = sorted(sequence, key=lambda month: month["T"])
        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)
        if values.shape != (seq_len, feature_count):
            raise ValueError("Monthly feature shape does not match the vocabulary.")
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")
        X[i] = values
        seen[i] = True
    if not seen.all():
        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")
    X.flush()


def input_fingerprints(X, metadata, features):
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    for start in range(0, len(X), 128):
        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {"model_input_sha256": tensor_hash.hexdigest(),
            "snapshot_manifest_sha256": digest_json(records),
            "feature_names_sha256": digest_json(features),
            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}


def load_inputs():
    from pyspark.sql import functions as F
    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid feature names, aliases or order.")
    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()
    frozen = read_sf("PATIENT_SPLIT").select(
        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()
    metadata, creation = validate_metadata_pair(source, frozen, features)
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")
    monthly = read_sf("TENSOR_MONTHLY")
    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
        raise ValueError("Monthly table columns differ from the frozen feature map.")
    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)
    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
        .agg(F.collect_list(F.struct(
            F.col("TIME_STEP").alias("T"),
            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
        )).alias("SEQUENCE"))
        .repartition(128))
    temporary, X = new_tensor_buffer(metadata, len(features))
    try:
        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
        hashes = input_fingerprints(X, metadata, features)
        X.flags.writeable = False
    except BaseException:
        X._mmap.close()
        temporary.cleanup()
        raise
    y = metadata.RESP.to_numpy(dtype=np.float32)
    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)
               for name in ("train", "validation", "test")}
    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)
    return {"X": X, "y": y, "metadata": metadata, "features": features,
            "indices": indices, "hashes": hashes, "split_creation": creation,
            "temporary_directory": temporary}


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")

REFERENCE_TRAINING_ARTIFACT_NAMES = {
    "checkpoint.pt", "training_summary.json", "training_history.csv", "training_history.png"
}


def bind_report_to_reference(report, artifacts, reference_run_id, stage):
    """Bind a source audit to the original completed run without loading PyTorch."""
    if (not isinstance(reference_run_id, str)
            or not re.fullmatch(r"[A-Z][A-Z0-9_]*", reference_run_id)):
        raise ValueError("REFERENCE_RUN_ID must be the original saved uppercase run identifier.")
    if not isinstance(artifacts, dict) or set(artifacts) != REFERENCE_TRAINING_ARTIFACT_NAMES:
        raise ValueError("The original reference run must contain its exact four training artifacts.")
    try:
        summary = json.loads(artifacts["training_summary.json"])
    except (TypeError, ValueError, UnicodeError) as error:
        raise ValueError("The original reference training summary is not valid JSON.") from error
    if (not isinstance(summary, dict) or summary.get("training_complete") is not True
            or summary.get("run_id") != reference_run_id):
        raise ValueError("The reference training run is incomplete or belongs to another RUN_ID.")
    hashes = summary.get("input_hashes")
    expected_hashes = {"model_input_sha256", "snapshot_manifest_sha256",
                       "feature_names_sha256", "split_config_sha256"}
    if (not isinstance(hashes, dict) or set(hashes) != expected_hashes
            or any(not isinstance(value, str) or not re.fullmatch(r"[0-9a-f]{64}", value)
                   for value in hashes.values())):
        raise ValueError("The original reference summary lacks valid full input fingerprints.")
    checkpoint = artifacts["checkpoint.pt"]
    if not isinstance(checkpoint, bytes) or not checkpoint:
        raise ValueError("The original reference checkpoint is missing or empty.")
    stages = {
        "preparation": ("reuse_and_validate_saved_v63_tensor",
                        ("model_input_sha256", "feature_names_sha256")),
        "split": ("reuse_original_patient_split",
                  ("snapshot_manifest_sha256", "split_config_sha256", "feature_names_sha256")),
    }
    if stage not in stages or not isinstance(report, dict) or report.get("mode") != stages[stage][0]:
        raise ValueError("Unexpected source audit stage for reference verification.")
    for field in stages[stage][1]:
        if report.get(field) != hashes[field]:
            raise ValueError(
                f"Current saved inputs differ from original {reference_run_id} on {field}. "
                "Restore the original saved inputs; matching population counts alone are insufficient.")
    bound = dict(report)
    bound["reference_run_id"] = reference_run_id
    bound["reference_checkpoint_sha256"] = hashlib.sha256(checkpoint).hexdigest()
    bound["reference_input_hashes"] = dict(hashes)
    bound["checks"] = dict(report.get("checks", {}), matches_original_reference_run=True)
    return bound


def validate_feature_mapping(mapping):
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices in the saved FEATURE_MAP.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid saved feature names, aliases or order.")
    return features, aliases


def original_snapshot_hash(metadata):
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP)]
               for r in metadata.itertuples()]
    return digest_json(records)


def validate_original_population(metadata, features):
    observed = (len(metadata), int(metadata.PATIENT_ID.nunique()),
                int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(
            "This handoff reuses the completed V63 cohort; "
            f"snapshots/patients/positives/features changed to {observed}.")


def make_preparation_report(X, metadata, features, source_prefix):
    # fill_tensor has already validated all rows, labels, timesteps and counts.
    validate_original_population(metadata, features)
    if tuple(X.shape) != (len(metadata), 12, len(features)):
        raise ValueError("The saved model-input shape must be snapshots x 12 x features.")
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    zero_months = 0
    zero_snapshots = 0
    for start in range(0, len(X), 128):
        block = np.asarray(X[start:start + 128], dtype="<f4")
        tensor_hash.update(block.tobytes(order="C"))
        empty_months = np.all(block == 0, axis=2)
        zero_months += int(empty_months.sum())
        zero_snapshots += int(np.all(empty_months, axis=1).sum())
    groups = {}
    for feature in features:
        group = feature.split("__", 1)[0] if "__" in feature else "OTHER"
        groups[group] = groups.get(group, 0) + 1
    return {
        "report_version": 1,
        "mode": "reuse_and_validate_saved_v63_tensor",
        "source_prefix": source_prefix,
        "source_tables": [source_prefix + "_" + suffix
                          for suffix in ("FEATURE_MAP", "SNAPSHOTS", "TENSOR_MONTHLY")],
        "n_snapshots": len(metadata),
        "n_patients": int(metadata.PATIENT_ID.nunique()),
        "n_positive_snapshots": int(metadata.RESP.sum()),
        "n_negative_snapshots": int(len(metadata) - metadata.RESP.sum()),
        "n_features": len(features),
        "n_timesteps": 12,
        "n_monthly_rows": int(X.shape[0] * X.shape[1]),
        "tensor_shape": list(X.shape),
        "model_input_dtype": "little_endian_float32",
        "minimum_snapshot_end_date": str(metadata.END_DT.min()),
        "maximum_snapshot_end_date": str(metadata.END_DT.max()),
        "zero_activity_months": zero_months,
        "zero_activity_snapshots": zero_snapshots,
        "feature_group_counts": groups,
        "model_input_sha256": tensor_hash.hexdigest(),
        "source_snapshots_sha256": original_snapshot_hash(metadata),
        "feature_names_sha256": digest_json(features),
        "checks": {
            "exact_saved_population": True,
            "unique_patient_date_keys": True,
            "exact_binary_labels": True,
            "exact_12_unique_timesteps": True,
            "complete_snapshot_coverage": True,
            "finite_nonnegative_counts": True,
            "zero_activity_sequences_preserved": True,
        },
        "scope": "Saved tensor validation only; raw claims and mappings were not regenerated.",
        "feature_selection": "Deferred to TRAIN-only processing in notebook 03.",
    }


def make_split_audit_report(metadata, creation, features, source_prefix):
    # validate_metadata_pair has already checked source coverage and all split rules.
    validate_original_population(metadata, features)
    expected = {
        "train": (16256, 8712, 941),
        "validation": (3481, 1867, 202),
        "test": (3414, 1868, 202),
    }
    summary = []
    patient_sets = {}
    for split_name in ("train", "validation", "test"):
        part = metadata.loc[metadata.SPLIT == split_name]
        observed = (len(part), int(part.PATIENT_ID.nunique()), int(part.RESP.sum()))
        if observed != expected[split_name]:
            raise ValueError(
                f"Saved {split_name} split differs from RUN_001: {observed}. "
                "Restore the original manifest; this notebook does not reshuffle patients.")
        patient_sets[split_name] = set(part.PATIENT_ID)
        summary.append({
            "split": split_name,
            "snapshots": observed[0],
            "patients": observed[1],
            "positive_snapshots": observed[2],
            "negative_snapshots": observed[0] - observed[2],
            "positive_fraction": observed[2] / observed[0],
            "minimum_snapshot_end_date": str(part.END_DT.min()),
            "maximum_snapshot_end_date": str(part.END_DT.max()),
        })
    overlaps = {
        "train_validation": len(patient_sets["train"] & patient_sets["validation"]),
        "train_test": len(patient_sets["train"] & patient_sets["test"]),
        "validation_test": len(patient_sets["validation"] & patient_sets["test"]),
    }
    if any(overlaps.values()):
        raise ValueError("A patient appears in more than one split.")
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {
        "report_version": 1,
        "mode": "reuse_original_patient_split",
        "source_prefix": source_prefix,
        "source_table": source_prefix + "_PATIENT_SPLIT",
        "split_summary": summary,
        "patient_overlap_counts": overlaps,
        "source_snapshots_sha256": original_snapshot_hash(metadata),
        "snapshot_manifest_sha256": digest_json(records),
        "feature_names_sha256": digest_json(features),
        "split_config_sha256": digest_json(creation),
        "saved_split_creation": creation,
        "new_assignments_created": False,
        "checks": {
            "source_keys_and_labels_match": True,
            "both_classes_in_each_split": True,
            "patient_overlap_zero": True,
            "original_split_counts_match": True,
            "feature_order_and_timesteps_match": True,
        },
    }


def compare_preparation_to_split(preparation_report, split_report):
    for report in (preparation_report, split_report):
        if (not report.get("reference_run_id") or not report.get("reference_checkpoint_sha256")
                or not isinstance(report.get("reference_input_hashes"), dict)):
            raise ValueError("Both source audits must be bound to the original reference training run.")
    for field in ("source_prefix", "source_snapshots_sha256", "feature_names_sha256",
                  "reference_run_id", "reference_checkpoint_sha256", "reference_input_hashes"):
        if preparation_report.get(field) != split_report.get(field):
            raise ValueError(f"Stage 01 preparation and frozen split disagree on {field}.")
    if preparation_report.get("mode") != "reuse_and_validate_saved_v63_tensor":
        raise ValueError("Unexpected preparation report; run notebook 01 from this delivery.")
    return True


In [ ]:
# CELL 3 — Load the saved feature map and source snapshots
for suffix in ("FEATURE_MAP", "SNAPSHOTS", "TENSOR_MONTHLY"):
    if not table_exists(PREFIX + "_" + suffix):
        raise FileNotFoundError(
            f"Missing completed V63 input: {PREFIX}_{suffix}. "
            "Restore the saved input; this notebook does not reconstruct raw claims.")
mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
features, aliases = validate_feature_mapping(mapping)
metadata = checked_metadata(read_sf("SNAPSHOTS").select(
    "PATIENT_ID", "END_DT", "RESP").toPandas())
validate_original_population(metadata, features)
print(f"Saved inputs: {len(metadata):,} snapshots; "
      f"{metadata.PATIENT_ID.nunique():,} patients; {len(features):,} features.")


In [ ]:
# CELL 4 — Validate all saved monthly sequences without needing a split
from pyspark.sql import functions as F
if "X" in globals() and hasattr(X, "_mmap") and not X._mmap.closed:
    X._mmap.close()
if "temporary" in globals():
    temporary.cleanup()
monthly = read_sf("TENSOR_MONTHLY")
if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
    raise ValueError("Monthly table columns differ from the saved feature map.")
grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
    .agg(F.collect_list(F.struct(
        F.col("TIME_STEP").alias("T"),
        F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                  .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
    )).alias("SEQUENCE"))
    .repartition(128))
print("Validating the full tensor in a temporary driver file (about 1.06 GiB).", flush=True)
temporary, X = new_tensor_buffer(metadata, len(features))
try:
    fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
    X.flags.writeable = False
except BaseException:
    X._mmap.close()
    temporary.cleanup()
    raise


In [ ]:
# CELL 5 — Fingerprint the full tensor and build a deterministic audit report
try:
    preparation_report = make_preparation_report(X, metadata, features, PREFIX)
    if not table_exists(REFERENCE_MODEL_TABLE):
        raise FileNotFoundError(
            f"Missing original saved model {REFERENCE_MODEL_TABLE}. "
            "Restore the completed RUN_001 training artifacts before preparing new experiments.")
    reference_artifacts = read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_TRAINING_ARTIFACT_NAMES)
    preparation_report = bind_report_to_reference(
        preparation_report, reference_artifacts, REFERENCE_RUN_ID, "preparation")
except BaseException:
    X._mmap.close()
    temporary.cleanup()
    raise
preparation_artifacts = {
    "preparation_report.json": canonical_json(preparation_report).encode("utf-8")
}
print("Complete tensor coverage, labels, finite counts and 12 timesteps verified.")
print("Tensor values and feature order match the original saved", REFERENCE_RUN_ID)
print("Snapshot dates are prediction cutoffs, not patient enrolment dates.")


In [ ]:
# CELL 6 — Inspect aggregate data coverage
display(pd.DataFrame([{
    "snapshots": preparation_report["n_snapshots"],
    "patients": preparation_report["n_patients"],
    "positive_snapshots": preparation_report["n_positive_snapshots"],
    "features": preparation_report["n_features"],
    "timesteps": preparation_report["n_timesteps"],
    "minimum_END_DT": preparation_report["minimum_snapshot_end_date"],
    "maximum_END_DT": preparation_report["maximum_snapshot_end_date"],
    "zero_activity_snapshots": preparation_report["zero_activity_snapshots"],
    "zero_activity_months": preparation_report["zero_activity_months"],
}]))
display(pd.DataFrame([
    {"feature_group": name, "feature_count": count}
    for name, count in sorted(preparation_report["feature_group_counts"].items())
]))
print("No patient IDs or individual feature values are saved in this audit report.")


In [ ]:
# CELL 7 — Save the preparation report in the new experiment namespace
save_artifacts(PREPARATION_TABLE, preparation_artifacts)
# An identical existing report is verified; differing artifacts are never overwritten.


In [ ]:
# CELL 8 — Verify saved preparation and release the temporary tensor
try:
    if read_artifacts(PREPARATION_TABLE, preparation_artifacts) != preparation_artifacts:
        raise ValueError("Preparation report read-back mismatch.")
    print("Preparation saved and verified. Continue with notebook 02.")
finally:
    X._mmap.close()
    temporary.cleanup()
    del X
